<a href="https://colab.research.google.com/github/gangadevig1067/Tanglish-sentiment-analyzer/blob/main/Tanglish_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate gradio scikit-learn

In [3]:
import torch
import numpy as np
import gradio as gr
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

# --- PART A: LOAD TANGILSH DATASET ---
print("1/4: Downloading Tanglish dataset...")
dataset = load_dataset("community-datasets/tamilmixsentiment")
labels = dataset["train"].features["label"].names

# --- PART B: LOAD MULTILINGUAL AI MODEL ---
print("2/4: Loading XLM-RoBERTa model...")
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels)
)

# Convert text into numbers the AI can read
def tokenize_data(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

tokenized_ds = dataset.map(tokenize_data, batched=True)
train_ds = tokenized_ds["train"].shuffle(seed=42).select(range(2000))
eval_ds = tokenized_ds["validation"].shuffle(seed=42).select(range(500))

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true, preds),
        "f1_macro": f1_score(y_true, preds, average="macro")
    }

# --- PART C: TRAIN THE MODEL ---
print("3/4: Training model on Tanglish comments (takes ~3 mins)...")
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics
)

trainer.train()

# --- PART D: LAUNCH LIVE DEMO LINK ---
print("4/4: Starting your live Gradio demo link...")
model.to("cpu")
model.eval()

id2label = {
    0: "Positive",
    1: "Negative",
    2: "Neutral",
    3: "Mixed Feelings",
    4: "Not-Tamil"
}

def predict_sentiment(text):
    if not text.strip():
        return {"Please enter some text": 1.0}
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
    return {id2label[i]: float(probs[i]) for i in range(len(id2label))}

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Type a Tanglish comment here (e.g., Padam semma mass bro!)..."
    ),
    outputs=gr.Label(num_top_classes=3, label="Predicted Sentiment"),
    title="Tanglish Code-Mixed Sentiment Analyzer",
    description="Prototype NLP model fine-tuned on Tamil-English code-mixed social media comments.",
    examples=[
        ["Padam semma mass bro, vera level!"],
        ["First half super, aana second half konjam lag."],
        ["Waste of money, total disappointment."]
    ]
)

demo.launch(share=True)


1/4: Downloading Tanglish dataset...


README.md:   0%|          | 0.00/9.35k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  507kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 60.3kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (17 outputs left out)

3/4: Training model on Tanglish comments (takes ~3 mins)...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.140145,1.059106,0.666000,0.159904
2,1.070646,1.023971,0.666000,0.159904


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

4/4: Starting your live Gradio demo link...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ef727b09df7365995.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
import gradio as gr
import torch

id2label = {
    0: "Positive",
    1: "Negative",
    2: "Neutral",
    3: "Mixed Feelings",
    4: "Not-Tamil"
}

def predict_sentiment(text):
    if not text.strip():
        return {"Please enter some text": 1.0}
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1)[0]
    return {id2label[i]: float(probs[i]) for i in range(len(id2label))}

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Type a Tanglish comment here (e.g., Padam semma mass bro!)..."
    ),
    outputs=gr.Label(num_top_classes=3, label="Predicted Sentiment"),
    title="Tanglish Code-Mixed Sentiment Analyzer",
    description="Prototype NLP model fine-tuned on Tamil-English code-mixed social media comments.",
    examples=[
        ["Padam semma mass bro, vera level!"],
        ["First half super, aana second half konjam lag."],
        ["Waste of money, total disappointment."]
    ]
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d1453b7590064f2f8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
